# Pilot Data Collection — Bluesky

This notebook performs a **pilot data collection** from Bluesky for the project on *The Devil Wears Prada 2*.

The goal is to evaluate whether the topic provides enough **posts, authors, metadata and replies** to support both content analysis and network analysis.

**Pipeline of this notebook**
1. Login with app password (loaded from `credentials.json`, not tracked by git)
2. Pilot search on a curated set of queries
3. Deduplication + temporal filtering
4. Per-query summary + descriptive statistics
5. Decision criteria (GO / scope-reduced / NO-GO)
6. Save pilot CSVs to `data/raw/`


## 1. Imports and login

In [ ]:
import json
import time
from datetime import datetime, timezone

import pandas as pd
from atproto import Client


In [ ]:
# Load Bluesky credentials from local file.
# The credentials file is gitignored and must NEVER be committed.
with open("../credentials.json", "r") as f:
    credentials = json.load(f)

handle = credentials["handle"]
app_password = credentials["app_password"]

client = Client()
client.login(handle, app_password)
print("Logged in as:", handle)


## 2. Pilot queries

The 12 queries below target four overlapping audiences identified in the research design:

- **Film audience** — direct mentions of the movie/franchise
- **Fashion audience** — Miranda Priestly, Anna Wintour, runway moments
- **Celebrity-culture audience** — cast names paired with `prada` disambiguator
- **Event-driven audience** — guest appearances tied to the press tour (Lady Gaga, Doechii)

The `prada` disambiguator on celebrity-name queries avoids generic-name noise (e.g. an "Anne Hathaway" query without context returns way too many unrelated posts).


In [ ]:
queries = [
    # Film — direct
    '"the devil wears prada 2"',
    '"devil wears prada 2"',
    "#devilwearsprada2",
    '"the devil wears prada"',

    # Fashion / characters
    '"miranda priestly"',
    '"anna wintour" prada',

    # Cast — disambiguated with `prada`
    '"meryl streep" prada',
    '"anne hathaway" prada',
    '"emily blunt" prada',
    '"stanley tucci" prada',

    # Press-tour / guest moments
    '"lady gaga" prada',
    '"doechii" prada',
]
print(f"{len(queries)} pilot queries ready.")


## 3. Time window

The movie was released on **1 May 2026**. We use a 3-month window centred around the release to capture late hype + opening weekend + post-release discourse.

If the pilot returns too few posts we will widen this window in the full collection notebook.


In [ ]:
START_DATE = pd.Timestamp("2026-03-01", tz="UTC")
END_DATE   = pd.Timestamp("2026-05-21", tz="UTC")

# ISO strings to pass to the Bluesky API (server-side filter)
SINCE_ISO = START_DATE.isoformat().replace("+00:00", "Z")
UNTIL_ISO = END_DATE.isoformat().replace("+00:00", "Z")

print("Window:", SINCE_ISO, "→", UNTIL_ISO)


## 4. Helper functions

`collect_search_posts` tries the server-side `since`/`until` filter first (fewer wasted calls), and **falls back automatically** to a no-time-filter call if the running version of `atproto` or the Bluesky API rejects those parameters. The client-side pandas filter in §6 catches anything that slips through.


In [ ]:
def parse_dt_utc(value):
    """Parse Bluesky `created_at` field into a UTC pandas Timestamp."""
    if value is None:
        return pd.NaT
    try:
        return pd.to_datetime(value, utc=True)
    except Exception:
        return pd.NaT


In [ ]:
def post_to_dict(post, query):
    """Flatten a Bluesky post object into a row dict."""
    record = post.record
    return {
        "uri": post.uri,
        "cid": post.cid,
        "query": query,
        "author_handle": post.author.handle,
        "author_did": post.author.did,
        "author_display_name": post.author.display_name,
        "text": getattr(record, "text", None),
        "created_at": getattr(record, "created_at", None),
        "reply_count": getattr(post, "reply_count", None),
        "repost_count": getattr(post, "repost_count", None),
        "like_count": getattr(post, "like_count", None),
        "quote_count": getattr(post, "quote_count", None),
        "indexed_at": getattr(post, "indexed_at", None),
        "lang": ",".join(getattr(record, "langs", []) or []),
    }


In [ ]:
def call_with_retries(fn, *, max_retries=3, base_sleep=2.0):
    """
    Call `fn` (a 0-arg callable that performs one API request) with
    exponential backoff. Re-raises if the final attempt still fails.
    """
    last_exc = None
    for attempt in range(max_retries):
        try:
            return fn()
        except Exception as exc:
            last_exc = exc
            wait = base_sleep * (2 ** attempt)
            print(f"  ! transient error ({exc.__class__.__name__}): "
                  f"retrying in {wait:.1f}s")
            time.sleep(wait)
    raise last_exc


In [ ]:
def _search_posts_request(client, params, *, use_time_filter):
    """One API call. If `use_time_filter` is False, strips since/until."""
    p = dict(params)
    if not use_time_filter:
        p.pop("since", None)
        p.pop("until", None)
    return client.app.bsky.feed.search_posts(params=p)


def collect_search_posts(
    client,
    query,
    *,
    since_iso=SINCE_ISO,
    until_iso=UNTIL_ISO,
    limit_total=100,
    page_limit=25,
    sleep_time=1.0,
):
    """
    Collect posts from Bluesky search for a given query.

    Uses cursor-based pagination. Tries server-side `since`/`until`
    filtering first; if the API or atproto version rejects those
    parameters, automatically falls back to no-time-filter mode and
    relies on the client-side pandas filter applied later.
    """
    posts = []
    cursor = None
    use_time_filter = True  # toggled off on first rejection

    while len(posts) < limit_total:
        params = {
            "q": query,
            "limit": min(page_limit, limit_total - len(posts)),
            "since": since_iso,
            "until": until_iso,
        }
        if cursor is not None:
            params["cursor"] = cursor

        try:
            response = call_with_retries(
                lambda: _search_posts_request(
                    client, params, use_time_filter=use_time_filter
                )
            )
        except Exception as exc:
            # If the failure looks parameter-related and we still have
            # the time filter on, retry once without it before giving up.
            msg = str(exc).lower()
            param_error = any(
                tok in msg for tok in ("since", "until", "invalid", "param", "validation")
            )
            if use_time_filter and param_error:
                print(f"  ⚠ server-side time filter rejected — "
                      f"falling back to client-side filter only")
                use_time_filter = False
                continue
            print(f"  ✗ giving up on query {query!r}: {exc}")
            break

        if not response.posts:
            break

        for post in response.posts:
            posts.append(post_to_dict(post, query))

        cursor = response.cursor
        if cursor is None:
            break

        time.sleep(sleep_time)

    return posts


## 5. Run the pilot

100 posts max per query, page size 25. With 12 queries this caps the pilot at 1 200 raw posts (before dedup).


In [ ]:
all_posts = []

for query in queries:
    print(f"→ {query}")
    query_posts = collect_search_posts(
        client=client,
        query=query,
        limit_total=100,
        page_limit=25,
        sleep_time=1.0,
    )
    print(f"  collected: {len(query_posts)}")
    all_posts.extend(query_posts)
    time.sleep(2)

df_posts = pd.DataFrame(all_posts)
print(f"\nTotal raw posts: {len(df_posts)}")
df_posts.head()


## 6. Deduplication and temporal filtering

In [ ]:
df_posts_unique = df_posts.drop_duplicates(subset="uri").copy()
df_posts_unique["created_dt"] = df_posts_unique["created_at"].apply(parse_dt_utc)

df_posts_filtered = df_posts_unique[
    (df_posts_unique["created_dt"] >= START_DATE) &
    (df_posts_unique["created_dt"] <= END_DATE)
].copy()

print("Raw posts:     ", len(df_posts))
print("Unique posts:  ", len(df_posts_unique))
print("Filtered posts:", len(df_posts_filtered))
print("Unique authors:", df_posts_filtered["author_handle"].nunique())

df_posts_filtered.head()


## 7. Per-query summary

In [ ]:
query_summary = (
    df_posts_filtered
    .groupby("query")
    .agg(
        n_posts=("uri", "count"),
        unique_authors=("author_handle", "nunique"),
        total_replies=("reply_count", "sum"),
        avg_replies=("reply_count", "mean"),
        total_likes=("like_count", "sum"),
        avg_likes=("like_count", "mean"),
    )
    .sort_values("n_posts", ascending=False)
)
query_summary


## 8. Top posts by reply count

In [ ]:
top_posts = df_posts_filtered.sort_values("reply_count", ascending=False)
top_posts[[
    "query", "author_handle", "text", "created_dt",
    "reply_count", "repost_count", "like_count", "uri",
]].head(20)


## 9. Language distribution

Sentiment + NER lexicons we will use downstream (AFINN, VADER, NRCLex, SpaCy `en_core_web_sm`) are English-only. The share of English posts here tells us how much of the pilot is usable as-is vs. how much needs filtering / language detection.


In [ ]:
lang_counts = (
    df_posts_filtered["lang"]
    .fillna("")
    .replace("", "(none)")
    .value_counts()
)
print(lang_counts.head(10))

share_en = (
    df_posts_filtered["lang"]
    .fillna("")
    .str.contains("en")
    .mean()
)
print(f"\nShare of posts tagged with 'en': {share_en:.1%}")


## 10. GO / NO-GO decision

The pilot caps each query at 100 posts, so the *raw* ceiling is 1 200 posts before dedup. Thresholds below are calibrated to this ceiling — they tell us whether the topic is healthy enough on Bluesky to justify scaling `limit_total` up to 500–1 000 per query in the full collection.

| Pilot outcome | Posts (filtered) | Unique authors | Action |
|---|---|---|---|
| **GO strong** | > 700 | > 300 | full scope, 4 sub-RQs, full collection at `limit_total=1000` |
| **GO acceptable** | 300 – 700 | 150 – 300 | full scope but watch the network density; reconsider temporal sub-RQ if reply rate is low |
| **Weak** | < 300 *or* < 150 authors | — | widen queries (extra titles below, longer window), or pivot topic |

The "queries hitting the 100 cap" counter is the *leading indicator*: if many queries saturate, the full collection will project well above the pilot numbers.


In [ ]:
n_posts = len(df_posts_filtered)
n_authors = df_posts_filtered["author_handle"].nunique()
n_replies_total = int(df_posts_filtered["reply_count"].fillna(0).sum())
n_posts_with_reply = int((df_posts_filtered["reply_count"].fillna(0) > 0).sum())
queries_hitting_cap = (query_summary["n_posts"] >= 100).sum()

print(f"Filtered posts:           {n_posts}")
print(f"Unique authors:           {n_authors}")
print(f"Total replies (declared): {n_replies_total}")
print(f"Posts with ≥1 reply:      {n_posts_with_reply}")
print(f"Queries hitting 100 cap:  {queries_hitting_cap} / {len(queries)}")

# Pilot-calibrated decision (max raw = 1200, max filtered ≈ 800–1000 after dedup)
if n_posts > 700 and n_authors > 300:
    decision = "GO strong — full scope, scale limit_total to 1000 in full run"
elif n_posts >= 300 and n_authors >= 150:
    decision = "GO acceptable — full scope, monitor network density"
else:
    if queries_hitting_cap >= len(queries) // 2:
        decision = (
            "GO projected — many queries hit the cap; "
            "full collection should clear thresholds, re-evaluate after full run"
        )
    else:
        decision = "Weak — widen queries (see extras below) or pivot topic"

print(f"\nPilot decision: {decision}")


### Extra queries to add in the full collection

These are kept out of the pilot to avoid burning the 100-post-per-query budget on lower-yield variants, but should be added in `01_full_collection_bluesky.ipynb`:

```python
extra_queries = [
    '"devil wears prada sequel"',
    '"prada sequel"',
    '"dwp2"',
    '"#dwp2"',
]
```


## 11. Save pilot data

In [ ]:
from pathlib import Path

out_dir = Path("../data/raw")
out_dir.mkdir(parents=True, exist_ok=True)

df_posts_filtered.to_csv(out_dir / "bluesky_posts_pilot.csv", index=False)
query_summary.to_csv(out_dir / "bluesky_query_summary_pilot.csv")

print("Saved:")
print(" -", out_dir / "bluesky_posts_pilot.csv")
print(" -", out_dir / "bluesky_query_summary_pilot.csv")


## 12. What to send back

Paste these four numbers + the `query_summary` table into the next chat so we can decide GO scope and start the full collection:

```python
print("Filtered posts:", len(df_posts_filtered))
print("Unique authors:", df_posts_filtered["author_handle"].nunique())
print("Total replies:", df_posts_filtered["reply_count"].sum())
print("Posts with ≥1 reply:", (df_posts_filtered["reply_count"] > 0).sum())
query_summary
```
